## Evaluación de modelos


### Modelo ingenuo

Añadimos la raíz del proeycto a la ruta de modulos ya que Jupyter arranca dentro de Notebooks y desde ahí no vel el paquete src

In [4]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

Reconstruimos el dataset a partir del csv y aplicamos la partición y la normalización, dejando el dataset listo para modelar.

In [6]:
from src.features import construir_dataset
from src.splits import particionar, normalizar

df = construir_dataset()
partes = particionar(df)
partes_norm, est = normalizar(partes)

[data] Caché local: GSPC_2000-01-01_2026-09-01.csv
[features] 6705 filas -> 6678 tras eliminar NaN
[features] Guardado en C:\Users\Victor\Desktop\Victor\Matematicas\neuro-fuzzy-volatility-forecasting\data\processed\dataset.csv
[splits] train  :  3998 filas  (2000-02-03 → 2015-12-23)
[splits] val    :  1001 filas  (2016-01-04 → 2019-12-23)
[splits] test   :  1669 filas  (2020-01-02 → 2026-08-24)
[splits] estres :    52 filas  (2020-02-18 → 2020-04-30)


**Evaluación del modelo ingenuo**

In [7]:
from src.evaluacion import evaluar, tabla_resultados
from src.modelos_ingenuos import predecir_ingenuo

# Usamos `partes`, no `partes_norm`: los ingenuos no se entrenan y
# trabajan en escala original.
test = partes["test"]
idx_estres = partes["estres"].index
y_test = test["y"]

filas = [evaluar(y_test, predecir_ingenuo(test, col), idx_estres, nombre)
         for col, nombre in [("rv_1", "Ingenuo 1d"),
                             ("rv_5", "Ingenuo 5d"),
                             ("rv_22", "Ingenuo 22d")]]

tabla = tabla_resultados(filas)
tabla.to_csv("../results/tables/e1_ingenuos.csv")
tabla

,RMSE,MAE,QLIKE,RMSE_tranq,MAE_tranq,QLIKE_tranq,RMSE_estres,MAE_estres,QLIKE_estres
modelo,,,,,,,,,
Ingenuo 22d,0.1094,0.0648,0.1211,0.0792,0.0549,0.1130,0.4347,0.3721,0.3725
Ingenuo 5d,0.1002,0.0637,0.1680,0.0846,0.0580,0.1619,0.3151,0.2413,0.3593
Ingenuo 1d,0.1410,0.0967,8.5075,0.1239,0.0902,8.7350,0.4008,0.2995,1.4337


Conclusiones de la evaluación:

1. El ingenuo de día 1 es inservible. Obtenemos un QLIKE de 8.5 frente a 0,12 y 0.17 de los otros. El problema es que con días con cierre casi plano, la predicción $\hat{y}$, al ser cociente, dispara el error. RMSE y MAE apenas lo penalizan, por lo que se muestra porque QLIKE es la métrica adecuada
2. El ranking depende de la métrica. El de 22 días gana en QLIKE (0,121 vs 0,168), pero el de 5 días gana en RMSE (0,100 vs 0,109) y en el periodo de estrés (RMSE 0,315 vs 0,435). Como conclusión, podemos sacar que la ventana larga (22) gana en estabilidad y evita errores grandes en calma,pero en situaciones de estres no funciona tan bien como la ventana de 5 días.
3. Se observa un patrón importante: si comparamos el RMSE global (22 días)  y el RMSE tranquilo, se observa que ambos son muy parecidos (0.1 frente a 0.11). Se podría decir que el modelo funciona correctamente. Pero en cambio en períodos de estres se obtiene un RSME de 0.43 , es decir esta multipliando por 4 su error.

De los puntos 2 y 3 extraemos una conclusión muy valiosa para el futuro. En nuestro estudio estamos intentando predecir el valor de la volatilidad. Esta predicción es especialmente útil, el la vida real, para predecir escenarios de estres con volatilidades altas y preparase ante ellas. Es por eso que nos interesa generar un modelo que se adapte muy bien ante estas sitauciones en vez de uno que prediga muy bien estados de calma pero falle mucho en situaciones de volatidilad extrema. COn esto podemos extraer que:
- Que es mejor una ventana corta de 5 días que una ventan larga, ya que la ventan larga predicira mejor sitauciones de calma pero la ventan corta es mejor en situaciones de estres
- Es importante valorar el modelo no solo en conjuntos globales, sino también en conjuntos de estres para optimizar la utilidad en la vida real